# Customer segmentation with RFM analysis

This notebook groups customers using a simple, rule-based RFM approach.
It uses non-cancelled transactions with a valid customer ID.

In [ ]:
# Import the functions we built for segmentation
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

sys.path.append(str(Path.cwd().parent / "src"))

from rfm_segmentation import (
    assign_segments,
    build_customer_segments,
    compute_rfm_metrics,
    describe_segment_rules,
    load_or_build_cleaned_dataset,
    plot_frequency_vs_monetary,
    plot_recency_vs_frequency,
    plot_segment_counts,
    plot_segment_revenue,
    score_rfm,
)

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
# Load or build the cleaned dataset from the processed folder
repo_root = Path.cwd()
cleaned_path = repo_root / "data" / "processed" / "cleaned_retail_data.csv"
raw_path = repo_root / "data" / "raw" / "Online_Retail.xlsx"

cleaned_df = load_or_build_cleaned_dataset(raw_path=raw_path, cleaned_path=cleaned_path)
print(f"Cleaned dataset rows: {len(cleaned_df)}")
cleaned_df.head()

In [ ]:
# Build RFM metrics from the cleaned data
rfm_df = compute_rfm_metrics(cleaned_df)
rfm_df = score_rfm(rfm_df)
rfm_df = assign_segments(rfm_df)

rfm_df.head()

In [ ]:
# Show the segment rules in simple language
print("Segment rules")
print("-" * 40)
for segment, rule in describe_segment_rules().items():
    print(f"{segment}: {rule}")

In [ ]:
# Create the final output and save it to the processed folder
output_path = repo_root / "data" / "processed" / "customer_segments.csv"
segments_df = build_customer_segments(cleaned_df, cleaned_path=cleaned_path, output_path=output_path)
print(f"Segments saved to: {output_path}")
segments_df.head()

In [ ]:
# Visual 1: Number of customers in each segment
plot_segment_counts(segments_df, output_path=repo_root / "data" / "processed" / "segment_counts.png")

In [ ]:
# Visual 2: Revenue generated by each segment
plot_segment_revenue(segments_df, output_path=repo_root / "data" / "processed" / "segment_revenue.png")

In [ ]:
# Visual 3: Recency versus Frequency
plot_recency_vs_frequency(segments_df, output_path=repo_root / "data" / "processed" / "recency_frequency.png")

In [ ]:
# Visual 4: Frequency versus Monetary value
plot_frequency_vs_monetary(segments_df, output_path=repo_root / "data" / "processed" / "frequency_monetary.png")

In [ ]:
# Print short business insights
segment_summary = segments_df.groupby("segment").agg(
    customers=("customerid", "count"),
    revenue=("monetary", "sum"),
).reset_index()

print("Business insights")
print("-" * 40)
for _, row in segment_summary.sort_values("revenue", ascending=False).iterrows():
    print(f"{row['segment']}: {row['customers']} customers, revenue {row['revenue']:,.2f}")

print("\nShort insight: The strongest customer groups are usually the most recent and most frequent buyers, so retention efforts should focus on them first.")